# Rows

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import correlate, correlation_lags
from pathlib import Path
import napari
import tkinter as tk
from tkinter import filedialog

# ==========================================
# 1. DISCRETIZED TFM STITCHER CLASS
# ==========================================
class TFMStitcher:
    def __init__(self, vol1, vol2, axis=2):
        self.vol1 = vol1.astype(np.float32)
        self.vol2 = vol2.astype(np.float32)
        self.axis = axis
        self.shift_index = 0
        self.best_score = -1

    def find_optimal_shift(self, num_slices=150, method='max', ignore_z=30):
        z_limit = self.vol1.shape[0]
        z_start, z_end = ignore_z, z_limit - ignore_z
        
        slice_axis = 0 
        dim_size = self.vol1.shape[slice_axis]
        indices = np.linspace(0, dim_size, num_slices + 1).astype(int)
        
        self.best_score = -1
        self.shift_index = 0 # Default to 0 if no match found

        for i in range(num_slices):
            idx_start, idx_end = indices[i], indices[i+1]
            sl = [slice(None)] * 3
            sl[0], sl[slice_axis] = slice(z_start, z_end), slice(idx_start, idx_end)
            
            sub_v1, sub_v2 = self.vol1[tuple(sl)], self.vol2[tuple(sl)]
            collapse_axes = tuple([ax for ax in range(3) if ax != self.axis])
            
            sig1 = np.max(sub_v1, axis=collapse_axes) if method == 'max' else np.mean(sub_v1, axis=collapse_axes)
            sig2 = np.max(sub_v2, axis=collapse_axes) if method == 'max' else np.mean(sub_v2, axis=collapse_axes)

            if np.std(sig1) < 1e-6 or np.std(sig2) < 1e-6: continue

            sig1_norm, sig2_norm = (sig1 - np.mean(sig1)), (sig2 - np.mean(sig2))
            corr = correlate(sig1_norm, sig2_norm, mode='full', method='fft')
            lags = correlation_lags(len(sig1), len(sig2), mode='full')
            
            # --- POSITIVE SHIFT CONSTRAINT ---
            # Mask out all lags that are less than 0
            corr[lags < 0] = -np.inf 
            
            denom = np.sqrt(np.sum(sig1_norm**2) * np.sum(sig2_norm**2))
            norm_corr = corr / (denom + 1e-10)
            
            p_idx = np.argmax(norm_corr)
            
            # Ensure the best peak found is actually a valid number (not -inf)
            if norm_corr[p_idx] > self.best_score:
                self.best_score, self.shift_index = norm_corr[p_idx], lags[p_idx]

        return self.shift_index

    def stitch(self, blend_mode='linear'):
        shift = int(round(self.shift_index))
        s1, s2 = self.vol1.shape, self.vol2.shape
        L1, L2 = s1[self.axis], s2[self.axis]
        v1_start, v2_start = (abs(shift), 0) if shift < 0 else (0, shift)
        total_len = max(v1_start + L1, v2_start + L2)
        
        stitched = np.zeros((*s1[:self.axis], total_len, *s1[self.axis+1:]), dtype=np.float32)
        sl1, sl2 = [slice(None)]*3, [slice(None)]*3
        sl1[self.axis], sl2[self.axis] = slice(v1_start, v1_start+L1), slice(v2_start, v2_start+L2)

        stitched[tuple(sl1)] = self.vol1
        
        inter_s, inter_e = max(v1_start, v2_start), min(v1_start + L1, v2_start + L2)
        if inter_e > inter_s:
            overlap = inter_e - inter_s
            # Create ramp matching the correct axis
            ramp_shape = [1, 1, 1]
            ramp_shape[self.axis] = overlap
            ramp = np.linspace(0, 1, overlap).reshape(ramp_shape)
            if shift < 0: ramp = 1.0 - ramp
            
            idx1, idx2 = [slice(None)]*3, [slice(None)]*3
            idx1[self.axis], idx2[self.axis] = slice(inter_s-v1_start, inter_e-v1_start), slice(inter_s-v2_start, inter_e-v2_start)
            stitched[tuple(idx1)] = (self.vol1[tuple(idx1)] * (1-ramp)) + (self.vol2[tuple(idx2)] * ramp)

        if v2_start + L2 > inter_e:
            end_sl = [slice(None)]*3
            end_sl[self.axis] = slice(inter_e, v2_start+L2)
            src_sl = [slice(None)]*3
            src_sl[self.axis] = slice(inter_e-v2_start, L2)
            stitched[tuple(end_sl)] = self.vol2[tuple(src_sl)]
            
        return stitched

# ==========================================
# 2. MAIN EXECUTION (PATHLIB HARDCODED)
# ==========================================
if __name__ == "__main__":
    # Define your base directory
    # (Update this to your actual folder)
    # Path configuration
    IN_DIR = Path.cwd().parent / 'DATA' / '2D_TFM_Data' / 'FeC_Smile_3MHz_04022026_Filtered'

    # List your specific filenames using Path objects
    selected = [
        IN_DIR / "FeC_40_6_filtered_3D_TFM.npy",
        IN_DIR / "FeC_40_5_filtered_3D_TFM.npy",
        IN_DIR / "FeC_40_4_filtered_3D_TFM.npy"
    ]

    # Check if files actually exist
    for p in selected:
        if not p.exists():
            print(f"CRITICAL ERROR: File not found -> {p}")
            exit()

    print(f"Found {len(selected)} volumes. Starting Sequential Stitch...")

    # Initialize the process with the first volume
    global_vol = np.load(selected[0])
    
    for i in range(1, len(selected)):
        next_file_path = selected[i]
        next_vol = np.load(next_file_path)
        
        # Safety Check: Dimensions must match on Y and Z
        if global_vol.shape[0] != next_vol.shape[0] or global_vol.shape[1] != next_vol.shape[1]:
            print(f"!! SKIPPING {next_file_path.name}: Dimension mismatch.")
            continue

        print(f"Merging Volume {i+1}/{len(selected)}: {next_file_path.name}")
        
        stitcher = TFMStitcher(global_vol, next_vol, axis=2)
        stitcher.find_optimal_shift(num_slices=300, ignore_z=30)
        
        print(f" -> Shift: {stitcher.shift_index} px | Score: {stitcher.best_score:.3f}")
        
        global_vol = stitcher.stitch(blend_mode='linear')
        
    # --- FINAL SAVING ---
    save_path = IN_DIR / "FINAL_ASSEMBLY_STITCHED.npy"
    np.save(save_path, global_vol)
    print(f"\nSUCCESS: Assembly saved to {save_path}")

    # Visualize
    viewer = napari.Viewer()
    clim = [0, np.percentile(global_vol, 99.9)]
    viewer.add_image(global_vol, name="Final Assembly", colormap="viridis", contrast_limits=clim)
    napari.run()

Found 3 volumes. Starting Sequential Stitch...
Merging Volume 2/3: FeC_40_5_filtered_3D_TFM.npy
 -> Shift: 90 px | Score: 0.793
Merging Volume 3/3: FeC_40_4_filtered_3D_TFM.npy
 -> Shift: 0 px | Score: 0.750

SUCCESS: Assembly saved to /Users/ottobruce-gardyne/Documents/Year4/GIP/signal-processing-G2066/DATA/2D_TFM_Data/FeC_Smile_3MHz_04022026_Filtered/FINAL_ASSEMBLY_STITCHED.npy


: 

# Grid

In [9]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import correlate, correlation_lags
from pathlib import Path
import napari

# ==========================================
# 1. CONSENSUS-BASED TFM STITCHER CLASS
# ==========================================
class TFMStitcher:
    def __init__(self, vol1, vol2, axis=2):
        self.vol1 = vol1.astype(np.float32)
        self.vol2 = vol2.astype(np.float32)
        self.axis = axis
        self.shift_index = 0
        self.best_score = -1

    def find_optimal_shift(self, grid=(60, 20), ignore_top=15, expected=0, tolerance=200):
        z_dim, y_dim, x_dim = self.vol1.shape
        z_start, z_end = ignore_top, z_dim
        
        tile_z = (z_end - z_start) // grid[0]
        tile_y = y_dim // grid[1]
        
        all_shifts = []
        all_weights = []
        
        for r in range(grid[0]):
            for c in range(grid[1]):
                zs, ze = z_start + (r * tile_z), z_start + ((r + 1) * tile_z)
                ys, ye = c * tile_y, (c + 1) * tile_y
                
                # Recency Bias: boost voting power of tiles at the 'new' end of the volume
                dist_from_edge = c / grid[1] 
                bias = 0.5 + (0.5 * dist_from_edge) 

                prof1 = np.max(self.vol1[zs:ze, ys:ye, :], axis=(0, 1))
                prof2 = np.max(self.vol2[zs:ze, ys:ye, :], axis=(0, 1))

                # SNR Gate
                if np.std(prof1) < 1e-5 or np.max(prof1) < np.max(self.vol1) * 0.1:
                    continue

                # ZNCC Normalization
                p1_n = (prof1 - np.mean(prof1)) / (np.std(prof1) + 1e-10)
                p2_n = (prof2 - np.mean(prof2)) / (np.std(prof2) + 1e-10)
                
                corr = correlate(p1_n, p2_n, mode='full')
                lags = correlation_lags(len(p1_n), len(p2_n), mode='full')
                
                # Broad Search Constraint & Directional Safeguard (lags >= 0)
                mask = (lags >= expected - tolerance) & (lags <= expected + tolerance)
                mask = mask & (lags >= 0) 
                if not np.any(mask): continue
                corr[~mask] = -np.inf 

                peak_idx = np.argmax(corr)
                weight = corr[peak_idx] * bias
                
                all_shifts.append(lags[peak_idx])
                all_weights.append(weight)

        if not all_shifts:
            self.shift_index = 0
            self.best_score = 0
            return 0

        # Weighted Consensus Vote
        lag_min, lag_max = np.min(all_shifts), np.max(all_shifts)
        bins = np.arange(lag_min, lag_max + 2) - 0.5
        counts, bin_edges = np.histogram(all_shifts, bins=bins, weights=all_weights)
        
        self.shift_index = int(bin_edges[np.argmax(counts)] + 0.5)
        self.best_score = np.max(counts) # Total consensus weight
        return self.shift_index

    def stitch(self, blend_mode='linear'):
        shift = int(round(self.shift_index))
        s1, s2 = self.vol1.shape, self.vol2.shape
        L1, L2 = s1[self.axis], s2[self.axis]
        
        v1_start, v2_start = (abs(shift), 0) if shift < 0 else (0, shift)
        total_len = max(v1_start + L1, v2_start + L2)
        
        stitched = np.zeros((*s1[:self.axis], total_len, *s1[self.axis+1:]), dtype=np.float32)
        
        sl1, sl2 = [slice(None)]*3, [slice(None)]*3
        sl1[self.axis], sl2[self.axis] = slice(v1_start, v1_start+L1), slice(v2_start, v2_start+L2)
        stitched[tuple(sl1)] = self.vol1
        
        inter_s, inter_e = max(v1_start, v2_start), min(v1_start + L1, v2_start + L2)
        if inter_e > inter_s:
            overlap = inter_e - inter_s
            ramp_shape = [1, 1, 1]; ramp_shape[self.axis] = overlap
            ramp = np.linspace(0, 1, overlap).reshape(ramp_shape)
            if shift < 0: ramp = 1.0 - ramp
            
            idx1, idx2 = [slice(None)]*3, [slice(None)]*3
            idx1[self.axis], idx2[self.axis] = slice(inter_s-v1_start, inter_e-v1_start), slice(inter_s-v2_start, inter_e-v2_start)
            stitched[tuple(idx1)] = (self.vol1[tuple(idx1)] * (1-ramp)) + (self.vol2[tuple(idx2)] * ramp)

        if v2_start + L2 > inter_e:
            end_sl = [slice(None)]*3
            end_sl[self.axis] = slice(inter_e, v2_start+L2)
            src_sl = [slice(None)]*3
            src_sl[self.axis] = slice(inter_e-v2_start, L2)
            stitched[tuple(end_sl)] = self.vol2[tuple(src_sl)]
            
        return stitched

# ==========================================
# 2. MAIN SEQUENTIAL EXECUTION (WITH TAIL-SLICING)
# ==========================================
if __name__ == "__main__":
    IN_DIR = Path.cwd().parent / 'DATA' / '2D TFM Data' / 'FeC Smile 3MHz HARD 04022026 Filtered'

    selected = [
        IN_DIR / "FeC_40_3_filtered_3D_TFM.npy",
        IN_DIR / "FeC_40_2_filtered_3D_TFM.npy",
        IN_DIR / "FeC_40_1_filtered_3D_TFM.npy"
    ]

    print(f"Found {len(selected)} volumes. Initializing Assembly...")
    global_vol = np.load(selected[0])
    
    for i in range(1, len(selected)):
        next_vol = np.load(selected[i])
        print(f"\nMerging {i+1}/{len(selected)}: {selected[i].name}")

        # --- TAIL-SLICING SAFEGUARD ---
        # Only look at the last 1.2x of one volume's length of the assembly
        # This prevents the algorithm from seeing (and stitching to) earlier images
        window_size = int(next_vol.shape[2] * 1.2)
        offset_removed = max(0, global_vol.shape[2] - window_size)
        active_tail = global_vol[:, :, offset_removed:]

        # Create stitcher instance using ONLY the tail
        stitcher = TFMStitcher(active_tail, next_vol, axis=2)
        
        # Calculate relative shift within the tail
        relative_shift = stitcher.find_optimal_shift(grid=(60, 20), ignore_top=15, tolerance=200)
        
        # Convert to global coordinate system
        stitcher.shift_index = offset_removed + relative_shift
        
        print(f" -> Local Shift: {relative_shift} px | Global Shift: {stitcher.shift_index} px")
        print(f" -> Consensus Strength: {stitcher.best_score:.2f}")
        
        # Re-initialize stitcher with full global_vol to perform the actual merge
        full_stitcher = TFMStitcher(global_vol, next_vol, axis=2)
        full_stitcher.shift_index = stitcher.shift_index
        global_vol = full_stitcher.stitch()
        
    # --- SAVE & VISUALIZE ---
    save_path = IN_DIR / "FINAL_PROTECTED_ASSEMBLY.npy"
    np.save(save_path, global_vol)
    print(f"\nSUCCESS: Assembly saved to {save_path.name}")

    viewer = napari.Viewer(title="Protected Sequential Stitching")
    clim = [0, np.percentile(global_vol, 99.9)]
    viewer.add_image(global_vol, name="Final Assembly", contrast_limits=clim, colormap="viridis")
    napari.run()

Found 3 volumes. Initializing Assembly...

Merging 2/3: FeC_40_2_filtered_3D_TFM.npy
 -> Local Shift: 167 px | Global Shift: 167 px
 -> Consensus Strength: 485.79

Merging 3/3: FeC_40_1_filtered_3D_TFM.npy
 -> Local Shift: 200 px | Global Shift: 327 px
 -> Consensus Strength: 2616.34

SUCCESS: Assembly saved to FINAL_PROTECTED_ASSEMBLY.npy
